In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

# ===== 0. 模型定義 =====
class F1Net(nn.Module):
    def __init__(self, cat_dims, num_num, emb_dim=8, hidden_dim=64):
        super().__init__()
        self.emb_layers = nn.ModuleList([nn.Embedding(dim, emb_dim) for dim in cat_dims])
        self.mlp = nn.Sequential(
            nn.Linear(len(cat_dims)*emb_dim + num_num, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, x_cat, x_num):
        embs = [emb_layer(x_cat[:, i]) for i, emb_layer in enumerate(self.emb_layers)]
        x = torch.cat(embs + [x_num], dim=1)
        return self.mlp(x).squeeze(-1)

# ===== 1. 讀取資料 =====
winners = pd.read_csv('./data/winners.csv')
drivers = pd.read_csv('./data/drivers_updated.csv')
teams = pd.read_csv('./data/teams_updated.csv')
laps = pd.read_csv('./data/fastest_laps_updated.csv')

# ===== 2. 日期與欄位預處理 =====
winners['Date'] = pd.to_datetime(winners['Date'])
winners['year'] = winners['Date'].dt.year
winners['race_order'] = winners.groupby('year')['Date'].rank(method='first').astype(int)

laps['year'] = laps['year'].astype(int)
laps = pd.merge(
    laps,
    winners[['Grand Prix', 'year', 'Date']].drop_duplicates(),
    on=['Grand Prix', 'year'],
    how='left'
)

def time_to_seconds(tstr):
    if pd.isnull(tstr) or tstr == '':
        return np.nan
    parts = str(tstr).split(':')
    if len(parts) == 2:
        return float(parts[0]) * 60 + float(parts[1])
    elif len(parts) == 3:
        return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
    else:
        return np.nan
laps['Time_sec'] = laps['Time'].apply(time_to_seconds)

# ===== 3. 特徵與標籤生成 =====
def generate_features(winners, drivers, laps):
    season_points = defaultdict(lambda: defaultdict(int))
    season_wins = defaultdict(lambda: defaultdict(int))
    team_points = defaultdict(lambda: defaultdict(int))
    records = []

    for year in sorted(winners['year'].unique()):
        races = winners[winners['year'] == year].sort_values('Date')
        for _, row in races.iterrows():
            race = row['Grand Prix']
            date = row['Date']
            round_num = row['race_order']
            season_drivers = drivers[drivers['year'] == year]

            for _, drow in season_drivers.iterrows():
                driver = drow['Driver']
                team = drow['Car']
                nationality = drow['Nationality']

                points_so_far = season_points[year][driver]
                wins_so_far = season_wins[year][driver]
                team_pts_so_far = team_points[year][team]

                prior_track_wins = winners[
                    (winners['Winner'] == driver) &
                    (winners['Grand Prix'] == race) &
                    (winners['Date'] < date)
                ]
                career_track_wins = len(prior_track_wins)

                relevant_laps = laps[
                    (laps['Driver'] == driver) &
                    (laps['Grand Prix'] == race) &
                    (laps['Date'] < date)
                ]
                if len(relevant_laps) > 0:
                    best_lap = relevant_laps['Time_sec'].min()
                    is_new_on_track = 0
                else:
                    best_lap = laps['Time_sec'].mean()
                    is_new_on_track = 1

                is_winner = int((row['Winner'] == driver) and (row['Car'] == team))

                records.append({
                    'Grand Prix': race,
                    'year': year,
                    'race_order': round_num,
                    'Date': date,
                    'Driver': driver,
                    'Team': team,
                    'Nationality': nationality,
                    'SeasonWinsSoFar': wins_so_far,
                    'SeasonPointsSoFar': points_so_far,
                    'CareerTrackWins': career_track_wins,
                    'DriverTrackBestLap': best_lap,
                    'IsNewOnTrack': is_new_on_track,
                    'TeamSeasonPointsSoFar': team_pts_so_far,
                    'is_winner': is_winner
                })

                if row['Winner'] == driver:
                    season_points[year][driver] += 25
                    season_wins[year][driver] += 1
                    team_points[year][team] += 25
    return pd.DataFrame(records)

df = generate_features(winners, drivers, laps)

# ===== 4. 編碼與標準化 =====
cat_cols = ['Grand Prix', 'Driver', 'Team', 'Nationality']
num_cols = ['year', 'race_order', 'SeasonWinsSoFar', 'SeasonPointsSoFar',
            'CareerTrackWins', 'DriverTrackBestLap', 'IsNewOnTrack', 'TeamSeasonPointsSoFar']

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

# ===== 5. Dataset 類別（含原始索引）=====
class F1DatasetWithIndex(Dataset):
    def __init__(self, df, cat_cols, num_cols):
        self.df = df.reset_index()
        self.X_cat = self.df[cat_cols].values.astype(np.int64)
        self.X_num = self.df[num_cols].values.astype(np.float32)
        self.y = self.df['is_winner'].values.astype(np.float32)
        self.original_idx = self.df['index'].values
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return [self.X_cat[i], self.X_num[i], self.y[i], self.original_idx[i]]

# ===== 6. Top-3 評估函式 =====
def evaluate_top3(df, model, cat_cols, num_cols, device='cpu', batch_size=512):
    dataset = F1DatasetWithIndex(df, cat_cols, num_cols)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    model.eval()
    preds_dict = defaultdict(list)

    with torch.no_grad():
        for X_cat, X_num, y, original_idxs in loader:
            X_cat = X_cat.to(device)
            X_num = X_num.to(device)
            logits = model(X_cat, X_num)
            scores = torch.sigmoid(logits).cpu().numpy()
            y = y.numpy()
            original_idxs = original_idxs.numpy()
            for score, label, orig_idx in zip(scores, y, original_idxs):
                race_key = (df.at[orig_idx, 'year'], df.at[orig_idx, 'Grand Prix'], df.at[orig_idx, 'Date'])
                preds_dict[race_key].append((score, label))

    top3_correct = 0
    total_races = len(preds_dict)

    for race_key, vals in preds_dict.items():
        vals_sorted = sorted(vals, key=lambda x: x[0], reverse=True)
        true_positions = [i for i, v in enumerate(vals_sorted) if v[1] == 1]
        if not true_positions:
            total_races -= 1
            continue
        true_pos = true_positions[0]
        if true_pos < 3:
            top3_correct += 1

    top3_accuracy = top3_correct / total_races if total_races > 0 else 0
    print(f"Top-3 accuracy: {top3_accuracy:.4f}")
    return top3_accuracy

# ===== 7. 時序動態訓練 =====
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

window_size_days = 365
validation_period_days = 90

df = df.sort_values('Date')
start_date = df['Date'].min()
end_date = df['Date'].max()

cat_dims = [df[col].max() + 1 for col in cat_cols]

while start_date + pd.Timedelta(days=window_size_days) < end_date:
    train_end = start_date + pd.Timedelta(days=window_size_days)
    val_start = train_end
    val_end = train_end + pd.Timedelta(days=validation_period_days)

    train_data = df[df['Date'] < train_end]
    val_data = df[(df['Date'] >= val_start) & (df['Date'] < val_end)]

    if len(train_data) == 0 or len(val_data) == 0:
        print(f"Skipped period {val_start.date()} to {val_end.date()} due to empty data.")
        start_date += pd.Timedelta(days=validation_period_days)
        continue

    # 時序資料檢查（確保無洩漏）
    assert train_data['Date'].max() < val_data['Date'].min(), "時序資料洩漏！"

    trainset = F1DatasetWithIndex(train_data, cat_cols, num_cols)
    valset = F1DatasetWithIndex(val_data, cat_cols, num_cols)
    trainloader = DataLoader(trainset, batch_size=256, shuffle=True)
    valloader = DataLoader(valset, batch_size=256, shuffle=False)

    model = F1Net(cat_dims, len(num_cols))
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(5):
        model.train()
        for X_cat, X_num, y, _ in trainloader:
            X_cat, X_num, y = X_cat.to(device), X_num.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X_cat, X_num)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()

    # 評估 Top-3 準確率
    evaluate_top3(val_data, model, cat_cols, num_cols, device=device)

    start_date += pd.Timedelta(days=validation_period_days)


KeyError: np.int64(0)